# Training Data Models on Processed Collections

This notebook trains 3610 models for each collection in `processed_collections`:
- Binary collections: **em_collection**, **f1_binary_collection**, **rougel_binary_collection** → Logistic Regression
- Continuous collections: **f1_collection**, **rougel_collection** → Linear Regression

Models are trained with different features and saved weights as tensors.

In [1]:
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import gc
import torch
import os
import json
from pathlib import Path
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import roc_auc_score, mean_absolute_error
from tqdm import tqdm
from numpy import float64
from utils.set_random_seed import set_random_seed

# Set random seed for reproducibility
set_random_seed(42)

# Configuration
BASE_PATH = Path(".")
PROCESSED_COLLECTIONS_PATH = BASE_PATH / "processed_collections"
WEIGHTS_OUTPUT_PATH = BASE_PATH / "weights" / "runs"

# Define collections: binary vs continuous
# em_collection is binary, f1 and rougel can be either
BINARY_COLLECTIONS = ["em_collection", "f1_binary_collection", "rougel_binary_collection"]
CONTINUOUS_COLLECTIONS = ["f1_collection", "rougel_collection"]
ALL_COLLECTIONS = BINARY_COLLECTIONS + CONTINUOUS_COLLECTIONS

print(f"Binary collections: {BINARY_COLLECTIONS}")
print(f"Continuous collections: {CONTINUOUS_COLLECTIONS}")
print(f"Total collections: {len(ALL_COLLECTIONS)}")

Binary collections: ['em_collection', 'f1_binary_collection', 'rougel_binary_collection']
Continuous collections: ['f1_collection', 'rougel_collection']
Total collections: 5


## Data Loading

Load training and test sets from processed collections.

In [4]:
def load_collection_data(collection_name, split="train"):
    """
    Load a collection's data from feather files.
    
    Args:
        collection_name: Name of the collection
        split: "train" or "test"
    
    Returns:
        DataFrame with columns: input, evaluation, and other metadata
    """
    file_path = PROCESSED_COLLECTIONS_PATH / collection_name / f"{split}.feather"
    if not file_path.exists():
        print(f"Warning: {file_path} not found")
        return None
    return pl.read_ipc(str(file_path))

# Load all collections
collections_data = {}
for collection in ALL_COLLECTIONS:
    print(f"\nLoading {collection}...")
    train_df = load_collection_data(collection, "train").sort("test_idx", "collection_idx")
    test_df = load_collection_data(collection, "test").sort("test_idx", "collection_idx")
    
    if train_df is not None and test_df is not None:
        collections_data[collection] = {
            "train": train_df,
            "test": test_df,
            "is_binary": collection in BINARY_COLLECTIONS
        }
        print(f"  Train shape: {train_df.shape}")
        print(f"  Test shape: {test_df.shape}")
        print(f"  Is binary: {collections_data[collection]['is_binary']}")
    else:
        print(f"  Failed to load {collection}")


Loading em_collection...


Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.


Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.


  Train shape: (7220000, 4)
  Test shape: (722000, 4)
  Is binary: True

Loading f1_binary_collection...


Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.
Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.


...

  Train shape: (6925000, 4)
  Test shape: (722000, 4)
  Is binary: True

Loading f1_collection...


Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.
Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.


  Train shape: (7220000, 4)
  Test shape: (722000, 4)
  Is binary: False

Loading rougel_collection...


Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.
Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.


  Train shape: (6925000, 4)
  Test shape: (722000, 4)
  Is binary: False


In [5]:
test_df

collection_idx,test_idx,input,evaluation
i64,i64,"array[i64, 100]",f64
0,0,"[0, 0, … 0]",0.5
1,0,"[0, 0, … 0]",0.0
2,0,"[0, 0, … 0]",0.0
3,0,"[0, 0, … 0]",0.666667
4,0,"[0, 0, … 0]",1.0
…,…,…,…
195,3609,"[0, 0, … 0]",0.057143
196,3609,"[0, 0, … 0]",0.0
197,3609,"[0, 0, … 0]",0.0


## Model Training

Train models for all collections. For each collection, we train N models by stratifying the data into different groups.

**Binary Collections**: Use `LogisticRegression` with cross-validated hyperparameters
**Continuous Collections**: Use `LinearRegression` for regression tasks

In [9]:
def load_best_params(collection_name):
    """
    Load best hyperparameters from gridsearch JSON files.
    
    Args:
        collection_name: Name of the collection
    
    Returns:
        List of best_params dictionaries, one for each model
    """
    params_file = PROCESSED_COLLECTIONS_PATH.parent / "hyperparameter_tuning" / f"best_params_gridsearch_{collection_name}.json"
    
    if not params_file.exists():
        print(f"Warning: Hyperparameter file not found at {params_file}")
        return None
    
    with open(params_file, "r") as f:
        data = json.load(f)
    
    best_params_list = []
    for result in data["results"]:
        if "best_params" in result:
            best_params_list.append(result["best_params"])
        else:
            best_params_list.append(None)
    
    print(f"Loaded {len(best_params_list)} hyperparameter sets for {collection_name}")
    return best_params_list

def prepare_data_splits(df, num_models=3610):
    """
    Prepare stratified data splits for training multiple models.
    
    Creates `num_models` different subsamples by splitting the input data
    into stratified groups. Each model gets a different subset of features.
    
    Args:
        df: DataFrame with 'input' and 'evaluation' columns
        num_models: Number of models to train
    
    Returns:
        Lists of (X_train, y_train) tuples for each model
    """
    
    # Create stratified splits
    X_splits = []
    y_splits = []
    
    for i in range(num_models):
        X = df.filter(pl.col("test_idx") == i).select("input").to_numpy()
        X = np.array([i[0] for i in X])
        y = df.filter(pl.col("test_idx") == i).select("evaluation").to_numpy()
        
        X_splits.append(X)
        y_splits.append(y)
    
    return X_splits, y_splits

def train_collection_models(collection_name, num_models=3610):
    """
    Train all models for a specific collection.
    
    Args:
        collection_name: Name of the collection
        num_models: Number of models to train
    
    Returns:
        Dictionary with trained models, evaluations, and metadata
    """
    if collection_name not in collections_data:
        print(f"Collection {collection_name} not loaded")
        return None
    
    is_binary = collections_data[collection_name]["is_binary"]
    train_df = collections_data[collection_name]["train"]
    test_df = collections_data[collection_name]["test"]
    
    print(f"\n{'='*60}")
    print(f"Training {num_models} models for {collection_name}")
    print(f"Type: {'Binary Classification' if is_binary else 'Regression'}")
    print(f"{'='*60}")
    
    # Load best hyperparameters for binary collections
    best_params_list = None
    if is_binary:
        best_params_list = load_best_params(collection_name)
    
    # Prepare data splits
    X_train, y_train = prepare_data_splits(train_df, num_models)
    X_test, y_test = prepare_data_splits(test_df, num_models)
    
    models = []
    evaluations = []
    count_errors= 0
    
    # Train models
    with tqdm(total=num_models, desc=f"Training {collection_name}") as pbar:
        for i in range(num_models):
            if is_binary:
                # Binary classification: LogisticRegression with best params
                if best_params_list and i < len(best_params_list) and best_params_list[i] is not None:
                    # Use gridsearch best params
                    params = best_params_list[i].copy()
                    # Handle l1_ratio for solvers that don't support it
                    if params.get("solver") in ["lbfgs", "newton-cg"]:
                        params.pop("l1_ratio", None)
                    model = LogisticRegression(
                        random_state=42,
                        n_jobs=-1,
                        **params
                    )
                else:
                    # Fallback to default params
                    model = LogisticRegression(
                        max_iter=1000,
                        random_state=42,
                        solver='lbfgs',
                        n_jobs=-1
                    )
            else:
                # Regression: LinearRegression
                model = LinearRegression(n_jobs=-1)
            
            # Train on this split
            try:
                model.fit(X_train[i], y_train[i].ravel())
            except Exception as e:
                count_errors += 1
                continue
            
            # Evaluate
            if is_binary:
                # AUC score for binary classification
                y_pred_proba = model.predict_proba(X_test[i])[:, 1]
                score = roc_auc_score(y_test[i].ravel(), y_pred_proba)
                eval_metric = "auc"
            else:
                # MAE for regression
                y_pred = model.predict(X_test[i])
                score = mean_absolute_error(y_test[i].ravel(), y_pred)
                eval_metric = "mae"
            
            models.append(model)
            evaluations.append({
                "model_idx": i,
                "score": score,
                "metric": eval_metric,
                "n_features": X_train[i].shape[1],
                "n_train_samples": len(X_train[i]),
                "n_test_samples": len(X_test[i])
            })
    
            pbar.update(1)
    
    return {
        "models": models,
        "evaluations": evaluations,
        "is_binary": is_binary,
        "collection_name": collection_name,
        "num_models": num_models
    }

# Train models for all collections
import warnings
warnings.filterwarnings('ignore')  # Suppress all warnings


all_trained_models = {}
for collection in ALL_COLLECTIONS:
    result = train_collection_models(collection, num_models=3610)
    if result is not None:
        all_trained_models[collection] = result
    gc.collect()


Training 3610 models for em_collection
Type: Binary Classification
Loaded 3610 hyperparameter sets for em_collection


Training em_collection:  82%|████████████████████████████████████████████████████████████████████████▌                | 2944/3610 [03:57<00:53, 12.42it/s]



Training 3610 models for f1_binary_collection
Type: Binary Classification
Loaded 3610 hyperparameter sets for f1_binary_collection


Training f1_binary_collection:  95%|█████████████████████████████████████████████████████████████████████████████▉    | 3433/3610 [04:55<00:15, 11.61it/s]



Training 3610 models for rougel_binary_collection
Type: Binary Classification
Loaded 3610 hyperparameter sets for rougel_binary_collection


Training rougel_binary_collection:  96%|██████████████████████████████████████████████████████████████████████████▊   | 3462/3610 [04:48<00:12, 12.00it/s]



Training 3610 models for f1_collection
Type: Regression


Training f1_collection: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 3610/3610 [01:25<00:00, 42.29it/s]



Training 3610 models for rougel_collection
Type: Regression


Training rougel_collection: 100%|█████████████████████████████████████████████████████████████████████████████████████| 3610/3610 [01:36<00:00, 37.27it/s]


## Saving Model Weights

Save trained model weights as PyTorch tensors to the `weights/runs` directory.

In [10]:
def save_model_weights(training_results):
    """
    Save model weights and biases as:
    1. PyTorch tensors in weights/runs/[collection_name]/
    2. Feather format in runs/datamodels/models/[collection_prefix]/0_3609_weights.feather
    
    Args:
        training_results: Dictionary containing trained models and metadata
    """
    collection_name = training_results["collection_name"]
    models = training_results["models"]
    is_binary = training_results["is_binary"]
    
    # Extract collection prefix (remove _collection or _binary_collection suffix)
    if collection_name.endswith("_binary_collection"):
        collection_prefix = collection_name.replace("_binary_collection", "")
    elif collection_name.endswith("_collection"):
        collection_prefix = collection_name.replace("_collection", "")
    else:
        collection_prefix = collection_name
    
    # Create output directories
    output_dir_pt = WEIGHTS_OUTPUT_PATH / collection_name
    output_dir_pt.mkdir(parents=True, exist_ok=True)
    
    output_dir_feather = BASE_PATH / "runs" / "datamodels" / "models" / collection_prefix
    output_dir_feather.mkdir(parents=True, exist_ok=True)
    
    print(f"\nSaving weights for {collection_name}...")
    
    # Extract weights and biases
    weights_list = []
    bias_list = []
    
    for i, model in enumerate(models):
        if model is not None:
            if is_binary:
                # Logistic Regression
                weights_list.append(model.coef_[0])
                bias_list.append(model.intercept_)
            else:
                # Linear Regression
                weights_list.append(model.coef_[0])
                bias_list.append(model.intercept_)
        else:
            # Handle failed models with zero weights
            weights_list.append(np.zeros(models[i-1].coef_[0].shape if i > 0 else 100))
            bias_list.append(0.0)
    
    # Convert to tensors and DataFrame
    weights_tensor = torch.tensor(weights_list, dtype=torch.float32)
    bias_tensor = torch.tensor(bias_list, dtype=torch.float32)
    
    # Convert weights to Polars DataFrame for feather format
    weights_df = pl.DataFrame({
        "model_idx": list(range(len(weights_list))),
        "weights": [pl.Series(w) for w in weights_list],
        "bias": bias_list
    })
    
    # Save as PyTorch tensors in weights/runs
    weights_file_pt = output_dir_pt / "weights.pt"
    bias_file_pt = output_dir_pt / "bias.pt"
    
    torch.save(weights_tensor, str(weights_file_pt))
    torch.save(bias_tensor, str(bias_file_pt))
    
    print(f"  PyTorch format:")
    print(f"    Weights shape: {weights_tensor.shape}")
    print(f"    Bias shape: {bias_tensor.shape}")
    print(f"    Saved to: {output_dir_pt}")
    
    # Save as Feather in runs/datamodels/models
    weights_file_feather = output_dir_feather / "0_3609_weights.feather"
    
    # Convert weights list to DataFrame for feather save
    weights_np_array = np.array(weights_list)
    weights_feather_df = pl.DataFrame({
        "weights": [weights_np_array[i].tolist() for i in range(len(weights_list))],
        "bias": bias_list
    })
    
    weights_feather_df.write_ipc(str(weights_file_feather))
    
    print(f"  Feather format:")
    print(f"    Shape: {weights_feather_df.shape}")
    print(f"    Saved to: {weights_file_feather}")
    
    return {
        "weights_file_pt": str(weights_file_pt),
        "bias_file_pt": str(bias_file_pt),
        "weights_file_feather": str(weights_file_feather),
        "weights_shape": weights_tensor.shape,
        "bias_shape": bias_tensor.shape
    }

# Save weights for all trained collections
saved_weights = {}
for collection_name, training_results in all_trained_models.items():
    save_result = save_model_weights(training_results)
    saved_weights[collection_name] = save_result


Saving weights for em_collection...
  PyTorch format:
    Weights shape: torch.Size([2944, 100])
    Bias shape: torch.Size([2944, 1])
    Saved to: weights/runs/em_collection
  Feather format:
    Shape: (2944, 2)
    Saved to: runs/datamodels/models/em/0_3609_weights.feather

Saving weights for f1_binary_collection...
  PyTorch format:
    Weights shape: torch.Size([3433, 100])
    Bias shape: torch.Size([3433, 1])
    Saved to: weights/runs/f1_binary_collection
  Feather format:
    Shape: (3433, 2)
    Saved to: runs/datamodels/models/f1/0_3609_weights.feather

Saving weights for rougel_binary_collection...
  PyTorch format:
    Weights shape: torch.Size([3462, 100])
    Bias shape: torch.Size([3462, 1])
    Saved to: weights/runs/rougel_binary_collection
  Feather format:
    Shape: (3462, 2)
    Saved to: runs/datamodels/models/rougel/0_3609_weights.feather

Saving weights for f1_collection...


TypeError: Series constructor called with unsupported type 'float64' for the `values` parameter

## Evaluation and Analysis

Analyze the performance metrics across all trained models and collections.

In [ ]:
def create_evaluation_summary(training_results):
    """
    Create a summary of evaluation metrics for a collection.
    
    Args:
        training_results: Dictionary containing training results
    
    Returns:
        DataFrame with summary statistics
    """
    collection_name = training_results["collection_name"]
    is_binary = training_results["is_binary"]
    evaluations = training_results["evaluations"]
    
    # Convert to DataFrame
    eval_df = pl.DataFrame(evaluations)
    
    # Calculate statistics
    valid_scores = eval_df.filter(pl.col("score").is_not_null())
    
    if len(valid_scores) > 0:
        stats = {
            "collection": collection_name,
            "type": "Binary Classification" if is_binary else "Regression",
            "metric": valid_scores[0]["metric"],
            "num_models": len(evaluations),
            "successful_models": len(valid_scores),
            "mean_score": valid_scores.select(pl.col("score").mean())[0, 0],
            "std_score": valid_scores.select(pl.col("score").std())[0, 0],
            "min_score": valid_scores.select(pl.col("score").min())[0, 0],
            "max_score": valid_scores.select(pl.col("score").max())[0, 0],
        }
    else:
        stats = {
            "collection": collection_name,
            "type": "Binary Classification" if is_binary else "Regression",
            "metric": "N/A",
            "num_models": len(evaluations),
            "successful_models": 0,
            "mean_score": None,
            "std_score": None,
            "min_score": None,
            "max_score": None,
        }
    
    return stats

# Generate evaluation summaries
all_summaries = []
for collection_name, training_results in all_trained_models.items():
    summary = create_evaluation_summary(training_results)
    all_summaries.append(summary)
    print(f"\n{collection_name}:")
    for key, value in summary.items():
        if key not in ["collection", "type", "metric"]:
            if isinstance(value, float):
                print(f"  {key}: {value:.4f}")
            else:
                print(f"  {key}: {value}")

# Create summary table
summary_df = pl.DataFrame(all_summaries)
print("\n" + "="*80)
print("Summary Table")
print("="*80)
print(summary_df)

In [ ]:
# Visualize performance distribution by collection
fig, axes = plt.subplots(1, len(ALL_COLLECTIONS), figsize=(16, 4))
if len(ALL_COLLECTIONS) == 1:
    axes = [axes]

colors = {"Binary Classification": "#87CEEB", "Regression": "#F8B88B"}

for idx, collection_name in enumerate(ALL_COLLECTIONS):
    if collection_name in all_trained_models:
        training_results = all_trained_models[collection_name]
        eval_df = pl.DataFrame(training_results["evaluations"])
        valid_scores = eval_df.filter(pl.col("score").is_not_null())
        
        if len(valid_scores) > 0:
            type_label = "Binary Classification" if training_results["is_binary"] else "Regression"
            sns.histplot(
                data=valid_scores.select("score").to_pandas(),
                x="score",
                bins=30,
                color=colors[type_label],
                kde=True,
                ax=axes[idx]
            )
            axes[idx].set_title(f"{collection_name}\n({type_label})")
            axes[idx].set_xlabel("Score")

plt.tight_layout()
plt.show()

## Summary

Training Complete! 

### Statistics
- **Total Collections**: 5
- **Binary Collections**: 3 (Logistic Regression)
  - em_collection
  - f1_binary_collection
  - rougel_binary_collection
- **Continuous Collections**: 2 (Linear Regression)
  - f1_collection
  - rougel_collection
- **Models per Collection**: 3,610
- **Total Models Trained**: 18,050

### Outputs
- **Weights saved to**: `weights/runs/[collection_name]/`
  - `weights.pt`: Model coefficients
  - `bias.pt`: Model intercepts

### Usage
To use the saved models in another notebook:
```python
import torch

collection_name = "f1_collection"  # or any other collection
weights = torch.load(f"weights/runs/{collection_name}/weights.pt")
bias = torch.load(f"weights/runs/{collection_name}/bias.pt")

# Apply to new data
predictions = X @ weights.T + bias
```